# Forecasting the Effect of Oil Price Increases on Financial Markets

Extended baseline notebook. Four parts:
1. **OLS return forecasting** — AR(1) vs ARX with oil, bond, MSCI controls
2. **HAR volatility model** — daily and weekly/monthly persistence components
3. **VAR model** — Granger causality + impulse response functions
4. **Monthly macro analysis** — oil effect on Industrial Production, ISM, Retail Sales

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 11, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

FIG_DIR = "../outputs/figures"

In [ ]:
def oos_metrics(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    mae  = np.mean(np.abs(y_true - y_pred))
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - ss_res / ss_tot
    return rmse, mae, r2

def run_ols(train, test, y_col, x_cols):
    X_tr = sm.add_constant(train[x_cols])
    X_te = sm.add_constant(test[x_cols])
    mdl = sm.OLS(train[y_col], X_tr).fit()
    pred = mdl.predict(X_te)
    return oos_metrics(test[y_col], pred), mdl

def fmt_table(df, title, fname=None):
    fig, ax = plt.subplots(figsize=(max(8, len(df.columns)*1.4), 0.8 + len(df)*0.5))
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=14)
    tbl = ax.table(cellText=df.values, colLabels=df.columns, rowLabels=df.index,
                   cellLoc='center', rowLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1, 1.6)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif c == -1:
            cell.set_facecolor('#ecf0f1')
            cell.set_text_props(fontweight='bold')
        else:
            cell.set_facecolor('#fdfefe' if r % 2 == 0 else '#f8f9f9')
    if fname:
        fig.savefig(f"{FIG_DIR}/{fname}")
    plt.show()

In [ ]:
# load daily data
df = pd.read_csv("../Data/interim/daily_clean.csv", parse_dates=["Date"]).set_index("Date")

# log-returns
df['r_oil']   = np.log(df['Brent futures'] / df['Brent futures'].shift(1))
df['r_sp500'] = np.log(df['S&P500'] / df['S&P500'].shift(1))
df['r_gold']  = np.log(df['Gold'] / df['Gold'].shift(1))
df['r_bond']  = np.log(df['US 10-year Rate'] / df['US 10-year Rate'].shift(1))
df['r_msci']  = np.log(df['MSCI World'] / df['MSCI World'].shift(1))
df['r_oil_pos'] = df['r_oil'].clip(lower=0)

print(f"Sample: {df.index[0].date()} to {df.index[-1].date()}, {len(df)} obs")

---
## Part 1 — OLS Return Forecasting (Daily)

Three models compared out-of-sample (80/20 split):
- **AR(1)**: own lag only
- **ARX-oil**: + lagged positive oil return
- **ARX-full**: + lagged bond return + lagged MSCI return

In [ ]:
# lagged regressors (all lagged by 1 day)
df['r_oil_pos_lag'] = df['r_oil_pos'].shift(1)
df['r_bond_lag']    = df['r_bond'].shift(1)
df['r_msci_lag']    = df['r_msci'].shift(1)

targets = {"Gold": "r_gold", "S&P500": "r_sp500", "US 10Y": "r_bond"}
for col in targets.values():
    df[f"{col}_lag"] = df[col].shift(1)

# train/test
d = df.dropna().copy()
s = int(len(d) * 0.8)
train_d, test_d = d.iloc[:s], d.iloc[s:]

# run models
results_ret = []
for name, y in targets.items():
    for label, xcols in [("AR(1)", [f"{y}_lag"]),
                          ("ARX-oil", [f"{y}_lag", 'r_oil_pos_lag']),
                          ("ARX-full", [f"{y}_lag", 'r_oil_pos_lag', 'r_bond_lag', 'r_msci_lag'])]:
        (rmse, mae, r2), _ = run_ols(train_d, test_d, y, xcols)
        results_ret.append({"Asset": name, "Model": label, "RMSE": f"{rmse:.6f}", "MAE": f"{mae:.6f}", "R2": f"{r2:.4f}"})

res = pd.DataFrame(results_ret)
tbl = res.pivot(index="Asset", columns="Model", values=["RMSE", "R2"])
tbl.columns = [f"{v} ({m})" for v, m in tbl.columns]
fmt_table(tbl, "Table 1 — Daily Return Forecasting (OOS)", "table1_daily_ret.png")

---
## Part 2 — HAR Volatility Model (Daily)

The Heterogeneous Autoregressive (HAR) model from Corsi (2009) decomposes volatility persistence into daily, weekly (5-day), and monthly (22-day) components:

$$\sigma^2_t = c + \beta_d \sigma^2_{t-1} + \beta_w \overline{\sigma^2}_{t-1:t-5} + \beta_m \overline{\sigma^2}_{t-1:t-22} + \varepsilon_t$$

We compare:
- **AR(1)**: own lagged vol only
- **HAR**: daily + weekly + monthly components
- **HAR + oil**: HAR + lagged oil volatility

In [ ]:
# volatility proxy = absolute return (more robust than squared)
for col_name in ['oil', 'sp500', 'gold', 'bond', 'msci']:
    r_col = f"r_{col_name}" if col_name != 'bond' else 'r_bond'
    if col_name == 'oil':
        r_col = 'r_oil'
    df[f'vol_{col_name}'] = df[r_col].abs()

# HAR components: daily (lag 1), weekly (avg of lags 1-5), monthly (avg of lags 1-22)
vol_targets = {"S&P500": "vol_sp500", "Gold": "vol_gold", "US 10Y": "vol_bond"}

for name, vc in vol_targets.items():
    df[f'{vc}_d']  = df[vc].shift(1)                        # daily component
    df[f'{vc}_w']  = df[vc].rolling(5).mean().shift(1)       # weekly component
    df[f'{vc}_m']  = df[vc].rolling(22).mean().shift(1)      # monthly component

# oil volatility components (lagged)
df['vol_oil_d'] = df['vol_oil'].shift(1)
df['vol_oil_w'] = df['vol_oil'].rolling(5).mean().shift(1)

# clean sample (need 22 lags for monthly component)
dv = df.dropna().copy()
sv = int(len(dv) * 0.8)
train_v, test_v = dv.iloc[:sv], dv.iloc[sv:]

# run HAR models
har_results = []
for name, vc in vol_targets.items():
    specs = [
        ("AR(1)",      [f'{vc}_d']),
        ("HAR",        [f'{vc}_d', f'{vc}_w', f'{vc}_m']),
        ("HAR + oil",  [f'{vc}_d', f'{vc}_w', f'{vc}_m', 'vol_oil_d', 'vol_oil_w']),
    ]
    for label, xcols in specs:
        (rmse, mae, r2), _ = run_ols(train_v, test_v, vc, xcols)
        har_results.append({"Asset": name, "Model": label, "RMSE": f"{rmse:.6f}", "MAE": f"{mae:.6f}", "R2": f"{r2:.4f}"})

har_df = pd.DataFrame(har_results)
tbl_har = har_df.pivot(index="Asset", columns="Model", values=["RMSE", "R2"])
tbl_har.columns = [f"{v} ({m})" for v, m in tbl_har.columns]
fmt_table(tbl_har, "Table 2 — HAR Volatility Model (OOS)", "table2_har_vol.png")

---
## Part 3 — VAR Model, Granger Causality and Impulse Responses

We estimate a VAR(p) on four return series: oil, S&P500, gold, US 10Y bond.
Then we test whether oil Granger-causes each financial market, and plot orthogonalized impulse responses to an oil shock.

In [ ]:
# prepare VAR data — oil ordered first (Cholesky ordering: oil is the shock source)
var_cols = ['r_oil', 'r_sp500', 'r_gold', 'r_bond']
var_data = df[var_cols].dropna()

# select lag order by AIC
model_var = VAR(var_data)
lag_selection = model_var.select_order(maxlags=10)
print("Lag order selection:")
print(lag_selection.summary())

best_lag = lag_selection.aic
print(f"\nSelected lag (AIC): {best_lag}")

In [ ]:
# estimate VAR
var_fit = model_var.fit(best_lag)
print(f"VAR({best_lag}) estimated with {var_fit.nobs} observations")
print(f"AIC: {var_fit.aic:.2f}, BIC: {var_fit.bic:.2f}")

### Granger Causality Tests

Does oil Granger-cause each financial market return? (and vice versa)

In [ ]:
# Granger causality tests
gc_rows = []
test_pairs = [
    ("r_oil", "r_sp500", "Oil -> S&P500"),
    ("r_oil", "r_gold",  "Oil -> Gold"),
    ("r_oil", "r_bond",  "Oil -> US 10Y"),
    ("r_sp500", "r_oil",  "S&P500 -> Oil"),
    ("r_gold", "r_oil",   "Gold -> Oil"),
    ("r_bond", "r_oil",   "US 10Y -> Oil"),
]

for causing, caused, label in test_pairs:
    test_result = var_fit.test_causality(caused, [causing], kind='f')
    gc_rows.append({
        "Direction": label,
        "F-stat": f"{test_result.test_statistic:.3f}",
        "p-value": f"{test_result.pvalue:.4f}",
        "Significant": "Yes" if test_result.pvalue < 0.05 else "No"
    })

gc_df = pd.DataFrame(gc_rows).set_index("Direction")
fmt_table(gc_df, "Table 3 — Granger Causality Tests (VAR)", "table3_granger.png")

### Impulse Response Functions

Orthogonalized IRFs: response of each market to a 1-std-dev oil shock (Cholesky, oil ordered first).

In [ ]:
# orthogonalized IRF — shock to oil, response of all variables
irf = var_fit.irf(periods=20)

# bootstrap confidence bands (Monte Carlo, 500 replications)
lower_band, upper_band = var_fit.irf_errband_mc(orth=True, repl=500, steps=20, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
response_names = ['r_sp500', 'r_gold', 'r_bond']
titles = ['S&P500', 'Gold', 'US 10Y Bond']
colors = ['#e74c3c', '#f39c12', '#2980b9']

shock_idx = var_cols.index('r_oil')

for ax, resp_name, title, color in zip(axes, response_names, titles, colors):
    resp_idx = var_cols.index(resp_name)
    irfs = irf.orth_irfs[:, resp_idx, shock_idx]

    ax.plot(range(21), irfs, color=color, linewidth=2)
    ax.fill_between(range(21),
                     lower_band[:, resp_idx, shock_idx],
                     upper_band[:, resp_idx, shock_idx],
                     alpha=0.15, color=color)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f"Response of {title}", fontweight='bold')
    ax.set_xlabel("Days after shock")
    ax.grid(alpha=0.2)

fig.suptitle("Figure 1 — Impulse Response to Oil Shock (Cholesky)", fontweight='bold', fontsize=13, y=1.03)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig1_irf_oil_shock.png", bbox_inches='tight')
plt.show()

---
## Part 4 — Monthly Macro Analysis

Using `monthly_clean.csv`: Industrial Production, Manufacturing ISM, US Retail Sales.
We merge monthly oil returns (resampled from daily) and test whether oil price increases predict macro activity.

In [ ]:
# load monthly macro data
macro = pd.read_csv("../Data/interim/monthly_clean.csv", parse_dates=["Date"]).set_index("Date")

# compute monthly oil return from daily data
oil_monthly = df['r_oil'].resample('ME').sum().rename('r_oil_m')
oil_monthly_pos = oil_monthly.clip(lower=0).rename('r_oil_pos_m')

# merge
macro = macro.join(oil_monthly).join(oil_monthly_pos)

# compute growth rates for macro variables (month-over-month % change)
macro['ip_growth']     = macro['Industrial production'].pct_change() * 100
macro['retail_growth'] = macro['US Retail Sales'].pct_change() * 100
macro['ism_chg']       = macro['Manufacturing ISM'].diff()

# lagged oil
macro['r_oil_pos_lag'] = macro['r_oil_pos_m'].shift(1)
macro['r_oil_lag']     = macro['r_oil_m'].shift(1)

# own lags
for col in ['ip_growth', 'retail_growth', 'ism_chg']:
    macro[f'{col}_lag'] = macro[col].shift(1)

# drop all rows with any NaN in the columns we use
use_cols = ['ip_growth', 'retail_growth', 'ism_chg', 'r_oil_pos_lag', 'r_oil_lag',
            'r_oil_m', 'r_oil_pos_m', 'ip_growth_lag', 'retail_growth_lag', 'ism_chg_lag']
macro = macro.dropna(subset=use_cols).copy()
macro = macro.replace([np.inf, -np.inf], np.nan).dropna(subset=use_cols).copy()
print(f"Monthly macro sample: {macro.index[0].date()} to {macro.index[-1].date()}, {len(macro)} obs")

In [ ]:
# macro forecasting: does lagged positive oil return predict macro activity?
macro_targets = {
    "IP Growth": "ip_growth",
    "Retail Sales Growth": "retail_growth",
    "ISM Change": "ism_chg",
}

sm_m = int(len(macro) * 0.8)
train_macro, test_macro = macro.iloc[:sm_m], macro.iloc[sm_m:]

macro_results = []
for name, y in macro_targets.items():
    lag = f"{y}_lag"
    for label, xcols in [("AR(1)", [lag]),
                          ("AR + oil", [lag, 'r_oil_pos_lag'])]:
        (rmse, mae, r2), mdl = run_ols(train_macro, test_macro, y, xcols)
        macro_results.append({"Target": name, "Model": label, "RMSE": f"{rmse:.4f}", "R2": f"{r2:.4f}"})

macro_df = pd.DataFrame(macro_results)
tbl_macro = macro_df.pivot(index="Target", columns="Model", values=["RMSE", "R2"])
tbl_macro.columns = [f"{v} ({m})" for v, m in tbl_macro.columns]
fmt_table(tbl_macro, "Table 4 — Monthly Macro Forecasting (OOS)", "table4_macro.png")

### In-sample coefficients — macro models with oil

In [ ]:
# in-sample coefficients for the AR + oil macro models
coef_rows = []
for name, y in macro_targets.items():
    lag = f"{y}_lag"
    X = sm.add_constant(train_macro[[lag, 'r_oil_pos_lag']])
    mdl = sm.OLS(train_macro[y], X).fit()
    for j, var in enumerate(mdl.params.index):
        coef_rows.append({
            "Target": name, "Variable": var,
            "Coeff": f"{mdl.params.values[j]:.4f}",
            "t-stat": f"{mdl.tvalues.values[j]:.2f}",
            "p-value": f"{mdl.pvalues.values[j]:.4f}",
            "": "***" if mdl.pvalues.values[j] < 0.01 else "**" if mdl.pvalues.values[j] < 0.05 else "*" if mdl.pvalues.values[j] < 0.1 else ""
        })

coef_macro = pd.DataFrame(coef_rows).set_index("Target")
fmt_table(coef_macro, "Table 5 — Macro Model Coefficients (In-Sample)", "table5_macro_coef.png")

### Monthly macro VAR — oil and IP growth

In [ ]:
# monthly VAR: oil return + IP growth + ISM change
macro_var_data = macro[['r_oil_m', 'ip_growth', 'ism_chg']].dropna()

macro_var = VAR(macro_var_data)
macro_lag = macro_var.select_order(maxlags=6).aic
macro_var_fit = macro_var.fit(macro_lag)

# Granger causality: does oil predict IP and ISM?
gc_macro = []
for caused, label in [('ip_growth', 'Oil -> IP Growth'), ('ism_chg', 'Oil -> ISM')]:
    t = macro_var_fit.test_causality(caused, ['r_oil_m'], kind='f')
    gc_macro.append({"Direction": label, "F-stat": f"{t.test_statistic:.3f}",
                     "p-value": f"{t.pvalue:.4f}", "Signif": "Yes" if t.pvalue < 0.05 else "No"})
for causing, label in [('ip_growth', 'IP Growth -> Oil'), ('ism_chg', 'ISM -> Oil')]:
    t = macro_var_fit.test_causality('r_oil_m', [causing], kind='f')
    gc_macro.append({"Direction": label, "F-stat": f"{t.test_statistic:.3f}",
                     "p-value": f"{t.pvalue:.4f}", "Signif": "Yes" if t.pvalue < 0.05 else "No"})

gc_macro_df = pd.DataFrame(gc_macro).set_index("Direction")
fmt_table(gc_macro_df, "Table 6 — Granger Causality: Oil vs Macro (Monthly VAR)", "table6_granger_macro.png")

In [ ]:
# monthly IRF — oil shock on IP and ISM
irf_macro = macro_var_fit.irf(periods=12)

# bootstrap confidence bands
lower_m, upper_m = macro_var_fit.irf_errband_mc(orth=True, repl=500, steps=12, seed=42)
macro_var_cols = ['r_oil_m', 'ip_growth', 'ism_chg']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, resp, title, color in zip(axes,
    ['ip_growth', 'ism_chg'], ['IP Growth', 'ISM Change'], ['#2980b9', '#27ae60']):
    ri = macro_var_cols.index(resp)
    si = macro_var_cols.index('r_oil_m')
    ax.plot(range(13), irf_macro.orth_irfs[:, ri, si], color=color, linewidth=2)
    ax.fill_between(range(13),
                     lower_m[:, ri, si],
                     upper_m[:, ri, si],
                     alpha=0.15, color=color)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f"Response of {title}", fontweight='bold')
    ax.set_xlabel("Months after shock")
    ax.grid(alpha=0.2)

fig.suptitle("Figure 2 — Monthly IRF: Oil Shock on Macro Variables", fontweight='bold', fontsize=13, y=1.03)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig2_irf_macro.png", bbox_inches='tight')
plt.show()

---
## Summary

| Method | What it shows |
|--------|--------------|
| **OLS baseline** (Table 1) | Oil alone adds almost nothing to daily return forecasts; bond + MSCI controls give ~0.3% RMSE gain |
| **HAR model** (Table 2) | Weekly/monthly persistence sharply improves vol forecasting; adding oil vol provides marginal further gain for S&P500 |
| **VAR + Granger** (Table 3) | Tests whether oil statistically predicts financial returns in a multivariate system |
| **IRF** (Figure 1) | Shows how a 1-std-dev oil shock propagates across markets over 20 days |
| **Macro forecasting** (Tables 4-5) | Tests oil's predictive power for IP growth, retail sales, ISM at monthly frequency |
| **Macro VAR + IRF** (Table 6, Figure 2) | Granger tests and impulse responses for oil vs macro variables |